# US ETF Data Update

Downloads nine US ETFs from Yahoo Finance starting 2011-10-01, using unadjusted OHLC and adjusted close data.

Output: `data/raw/etf_daily.csv`.

The latest unfinished US trading session is excluded. The previous data file is backed up before replacement.

CPI actual and forecast data are maintained separately in `data/raw/cpi_yoy.csv`.


In [ ]:
import sys
from pathlib import Path

# Locate the project root from the root itself or any subfolder, then share 00_engine via portfolio_accounting.
ROOT = Path.cwd().resolve()
while not (ROOT / 'pipeline' / '00_engine.ipynb').is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from portfolio_accounting import ENGINE as _engine, load_definitions

import shutil
import pandas as pd

TICKS = _engine['TICKS']


In [ ]:
DATA_DIR = ROOT / 'data' / 'raw'
ETF_CSV = DATA_DIR / 'etf_daily.csv'



def drop_unfinished(df, tz, close_minutes, label):
    """Drop today's row if it was downloaded before the local close (intraday prices)."""
    now = pd.Timestamp.now(tz=tz)
    if len(df) and df.index[-1].date() >= now.date() and now.hour * 60 + now.minute < close_minutes:
        print(f'{label}: dropped unfinished session {df.index[-1].date()} (local time {now:%H:%M}, market not yet closed)')
        df = df.iloc[:-1]
    return df


def has(data, t):
    return ('Adj Close', t) in data and not data['Adj Close'][t].dropna().empty


def download_all(start='2011-10-01'):
    import yfinance as yf
    data = yf.download(TICKS, start=start, auto_adjust=False)
    if data.empty:
        raise ValueError("US ETF download returned no data.")
    missing = [t for t in TICKS if not has(data, t)]
    if missing:
        raise ValueError(f"Missing US tickers: {missing}")
    return data


def build_etf_daily(data):
    return drop_unfinished(data, 'America/New_York', 16 * 60 + 30, 'etf_daily (US equities)')




def overwrite(df, target):
    if target.is_file():
        shutil.copy2(target, target.with_name(target.stem + '_backup.csv'))
    df.to_csv(target)


def main(download=True):
    if not download:
        print("Download skipped.")
        return None

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    data = download_all()
    etf = build_etf_daily(data)

    overwrite(etf, ETF_CSV)
    last_us = etf['Adj Close'][TICKS].dropna().index.max().date()
    print(f"Updated US ETF data through {last_us}")

    return {"etf_daily": etf}


In [ ]:
DOWNLOAD = True      # set False to skip the download
results = main(download=DOWNLOAD)
